In [22]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
import numpy as np
import datetime
import matplotlib.pyplot as plt

THRESHOLD_TIMESTAMPS = 16

In [3]:
def extract_json(filename: str):
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]
total_lenght = extract_json("train.jsonl")



In [ ]:
sessions = []

for i, (session_id, eventstotal) in enumerate(extract_json("train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 100000:
        break

In [5]:
class OttoDataSetSession(Dataset):
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [9]:
sessions_in_dataset = OttoDataSetSession(sessions)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]["timestamps"]
    if len(sample) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght.append(sample)




Total len of the Sessions: 100001


In [65]:
avg_ofs_day_dataset = []
for i in range(len(session_sample_lenght)):
    ts = session_sample_lenght[i]
    min_ts = min(ts)
    max_ts = max(ts)
    avg = sum(ts) / len(ts)
    diff_seconds = float(max_ts - min_ts) / 1000
    
    avg_ofs_day_dataset.append(diff_seconds)
    diff = datetime.timedelta(seconds=diff_seconds)
    
    
    #print(f"This is the min TimeStamp of this Session {i} {(datetime.datetime.fromtimestamp(int(min_ts / 1000)).strftime('%d-%m-%Y %H:%M:%S'))}")
    #print(f"This is the max TimeStamp of this Session {i} {(datetime.datetime.fromtimestamp(int(max_ts) / 1000)).strftime('%d-%m-%Y %H:%M:%S')}")
    #print(f"This is the avg TimeStamp of this Session {i} {(datetime.datetime.fromtimestamp(int(avg) / 1000)).strftime('%d-%m-%Y %H:%M:%S')}")
    
    #print(f"Difference of days and everything {diff}")
    #print("=======================================================================")
    
    if i == 10000:
        break

In [64]:
print(avg_ofs_day_dataset)
converted = [datetime.timedelta(seconds=x) for x in avg_ofs_day_dataset]
print(f"Total Avarage per 1000 of all the sessions {np.mean(converted)}")


[2380183.682, 2410054.967, 2409415.621, 1804866.676, 2281881.184, 2244731.623, 1233718.185, 1701577.186, 2380231.611, 2403935.839, 2320620.328, 2149093.004, 867444.175, 2411566.924, 2066874.002, 2361110.196, 1821217.655, 2418015.354, 1506709.357, 2330195.481, 1814946.439, 41722.683, 1818918.495, 2138376.777, 1980462.869, 2405573.941, 2367666.498, 2419190.714, 591566.11, 2393135.931, 2417279.076, 2419125.722, 663102.747, 2297317.898, 2332063.667, 2270407.952, 2066373.772, 2072974.68, 2308192.67, 1433824.269, 2339427.294, 2241574.907, 2127607.17, 2405571.054, 2408310.348, 2396319.716, 519192.932, 1090.975, 2237257.913, 1553889.496, 49133.133, 1874963.433, 2228567.622, 4178.741, 2126553.792, 2288458.635, 1180683.412, 297936.503, 562446.816, 2416069.951, 1850377.839, 2150214.826, 2326632.23, 2330568.715, 2330544.795, 2212497.0, 2035113.574, 1861213.949, 2413908.145, 1269103.762, 2067192.786, 2418081.61, 2389591.535, 2413155.021, 2411686.14, 77857.463, 1980573.333, 2402712.751, 2206913.114,